# DATA266 Lab 1 — Task 2: Yelp Polarity Sentiment Classification (Dipin)

Three models, all with **word embeddings learned from scratch** (no pretrained embeddings or LMs):

| Model | Architecture | What changes vs previous |
|---|---|---|
| Baseline | Embedding(200) → 1-layer LSTM(128) → last hidden state → linear | — |
| Experimental 1 | Embedding(200) → 2-layer **BiLSTM**(128) → concat final fwd/bwd → linear | direction + depth |
| Experimental 2 | Embedding(200) → 2-layer BiLSTM(128) → **additive attention pooling** → linear | pooling only |

Split/test protocol is shared with Khushi (stratified 90/10 of the official train pool, seed 42; official 38k test set untouched), so the team comparison is on identical data.
All hyperparameters live in `configs/task2_config.json`.

Set the environment variable `DIPIN_T2_SMOKE=1` to run this notebook on a tiny subset (pipeline check only).

In [ ]:
import json, os, platform, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Locate this member's task folder without any hard-coded personal path.
TASK_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "task2_config.json").exists())
sys.path.insert(0, str(TASK_DIR / "src"))

from data_prep import audit_raw, load_config, load_raw, prepare_all, set_seed, Preprocessor
from models import count_parameters
from train import get_device, hardware_info, load_model, make_loader, predict, train_model
import evaluate as ev

SMOKE = os.environ.get("DIPIN_T2_SMOKE") == "1"
cfg = load_config()
set_seed(cfg["seed"])
OUT = TASK_DIR / "outputs" / ("smoke" if SMOKE else "")  # smoke runs never touch real outputs
for sub in ["eda", "plots", "predictions", "error_review"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
device = get_device()
HW = hardware_info(device)
print("task dir:", TASK_DIR.name, "| smoke:", SMOKE)
print(json.dumps(HW, indent=2))

## 2.1 Data preprocessing
### Load raw data, audit for missing / malformed entries

In [ ]:
train_text, train_y, test_text, test_y = load_raw(cfg["dataset"])
audit = {"train": audit_raw(train_text, train_y), "test": audit_raw(test_text, test_y)}
pd.DataFrame(audit)

**Findings.** There are no missing texts, empty reviews, invalid labels or HTML tags. The only malformed content is *literal* escape sequences: about half of all reviews contain `\n` stored as two characters rather than a newline.
Left alone, punctuation stripping would turn `\n` into a stray `n` token, so the cleaner replaces these sequences with spaces first.
A few reviews become empty after cleaning (see the preprocessing table below). They are kept, not dropped, so the split stays identical to the team's, and the model handles them as length-0 inputs.

In [ ]:
counts = pd.DataFrame({"train": pd.Series(train_y).value_counts().sort_index(),
                       "test": pd.Series(test_y).value_counts().sort_index()})
counts.index = ["negative (0)", "positive (1)"]
counts["train_%"] = 100 * counts["train"] / counts["train"].sum()
counts["test_%"] = 100 * counts["test"] / counts["test"].sum()
display(counts)

wc_train = np.array([len(t.split()) for t in train_text])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(counts.index, counts["train"], color=["#c0504d", "#4f81bd"])
axes[0].set_title("Class distribution (train pool, 560k)")
for lab, name, c in [(0, "negative", "#c0504d"), (1, "positive", "#4f81bd")]:
    axes[1].hist(np.clip(wc_train[train_y == lab], 0, 800), bins=80, alpha=0.55, label=name, color=c)
axes[1].set_title("Review length (words, clipped at 800)"); axes[1].set_xlabel("words"); axes[1].legend()
fig.tight_layout(); fig.savefig(OUT / "eda" / "class_and_length_distribution.png", dpi=120); plt.show()

length_stats = pd.DataFrame({name: pd.Series(wc_train[train_y == lab]).describe(percentiles=[.5, .9, .95, .99])
                             for lab, name in [(0, "negative"), (1, "positive")]}).T
display(length_stats.round(1))

The classes are perfectly balanced (50/50 in both train and test), so no class weighting or resampling is needed, and accuracy is a meaningful headline metric.
Review lengths are heavily right-skewed. Negative reviews tend to be longer than positive ones (see table), so length is a weak confounder. That is why length slices are part of the robustness evaluation.

### Text preprocessing → tokenization → vocabulary → integer sequences
Pipeline: unescape literal `\n`, lowercase, expand negation contractions (`didn't` → `did not`), remove punctuation and special characters, whitespace-tokenize,
remove NLTK English stopwords **except negations** (`not`, `no`, `never`, … flip sentiment), then apply **WordNet lemmatization** (noun pass, then verb pass: `loved` → `love`, `places` → `place`).
Lemmatization rather than stemming keeps real words, so the learned embeddings and the error review stay interpretable.

In [ ]:
pre = Preprocessor(cfg["preprocessing"])
for t in [train_text[0], train_text[3]]:
    print("RAW :", t[:300].replace("\\n", " "))
    print("PROC:", " ".join(pre(t))[:300]); print()

t0 = time.time()
data = prepare_all(cfg, max_train=4000, out_dir=OUT / "data") if SMOKE else prepare_all(cfg)
print(f"preprocessing took {time.time() - t0:.0f}s")
meta = data["meta"]
vocab_size = len(data["stoi"])
display(pd.DataFrame(meta["splits"]).T)
print("vocab size (incl. <PAD>, <UNK>):", vocab_size, "| unique train tokens:", meta["train_unique_tokens"],
      "| train/val overlap:", meta["train_validation_overlap"])

In [ ]:
proc_len = data["lengths"]["train"]
fig, ax = plt.subplots(figsize=(7, 3.5))
lens_all = np.array(meta["splits"]["train"]["processed_length_percentiles_50_90_95_99"])
ax.hist(np.minimum(proc_len, cfg["preprocessing"]["max_sequence_length"]), bins=100, color="#4f81bd")
ax.axvline(cfg["preprocessing"]["max_sequence_length"], color="k", ls="--", label="max_len = 200")
ax.set_title("Processed sequence length (tokens after cleaning, capped)"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "eda" / "processed_length.png", dpi=120); plt.show()
print("processed length p50/p90/p95/p99:", lens_all,
      f"| truncated: {meta['splits']['train']['truncated_pct']:.2f}% of train")

`max_len = 200` covers about 96% of reviews without truncation, and keeps LSTM compute bounded.
For the ~4% that are longer, the encoder keeps the first 150 and the **last 50** tokens (head+tail), because Yelp reviews often end with the overall verdict.
The vocabulary is built on the **training split only**: tokens with frequency ≥ 3, capped at 40k. Everything else maps to `<UNK>`, and `<PAD>`=0 is masked out of the LSTM via packing.

### Embeddings learned from scratch
Each model owns an `nn.Embedding(vocab_size, 200, padding_idx=0)`, initialized uniformly in [-0.05, 0.05] and trained jointly with the classifier.
Nothing pretrained is loaded. Section 2.2 ends with a nearest-neighbour check showing the embeddings learned sentiment structure.

## 2.2 Model training
All three share: AdamW (lr 2e-3, wd 1e-4), batch 256, linear warm-up (5%) then cosine decay, gradient clipping at 1.0, BCE-with-logits loss on one logit,
up to 4 epochs with early stopping on validation macro-F1 (patience 2). The checkpoint with the best validation macro-F1 is kept.

In [ ]:
summaries, ckpts = {}, {}
for name in cfg["models"]:
    summaries[name], ckpts[name] = train_model(name, cfg, data, vocab_size, device,
                                               epochs=1 if SMOKE else None, tag="_smoke" if SMOKE else "")
ev.plot_history(summaries, OUT / "plots" / "training_curves.png")
from IPython.display import Image
Image(filename=str(OUT / "plots" / "training_curves.png"))

In [ ]:
eff = pd.DataFrame({n: {"parameters": s["parameter_count"], "best_epoch": s["best_epoch"], "epochs_run": s["epochs_run"],
                        "train_time_s": round(s["training_seconds"], 1),
                        "train_examples_per_sec": round(s["train_examples_per_sec"], 1),
                        "peak_gpu_mem_MB": None if s["peak_accelerator_bytes"] is None else round(s["peak_accelerator_bytes"] / 2**20, 1),
                        "peak_host_rss_MB": round(s["peak_host_rss_bytes"] / 2**20, 1),
                        "gpu": s["hardware"].get("gpu"), "cpu": s["hardware"]["cpu"],
                        "checkpoint": s["checkpoint"], "raw_log": s["raw_log"]} for n, s in summaries.items()}).T
eff

## Evaluation on the official 38,000-review test set

In [ ]:
test_loader = make_loader(data["ids"]["test"], data["lengths"]["test"], data["labels"]["test"], 1024, False, cfg["seed"])
slices = ev.build_slices(data["raw_text"]["test"], data["unks"]["test"], data["lengths"]["test"], data["truncated"]["test"])
metrics, preds, probs, per_slice = {}, {}, {}, {}
for name, path in ckpts.items():
    model, _ = load_model(path, device)
    t0 = time.perf_counter()
    y_test, probs[name] = predict(model, test_loader, device)
    infer_s = time.perf_counter() - t0
    metrics[name], preds[name] = ev.compute_metrics(y_test, probs[name], cfg)
    metrics[name].update(parameter_count=summaries[name]["parameter_count"],
                         training_time_s=summaries[name]["training_seconds"],
                         train_examples_per_sec=summaries[name]["train_examples_per_sec"],
                         inference_examples_per_sec=len(y_test) / infer_s,
                         peak_gpu_memory_bytes=summaries[name]["peak_accelerator_bytes"],
                         peak_host_rss_bytes=summaries[name]["peak_host_rss_bytes"])
    per_slice[name] = ev.slice_metrics(y_test, preds[name], slices)
    ev.plot_confusion(y_test, preds[name], name, OUT / "plots" / f"confusion_{name}.png")
    pd.DataFrame({"test_index": np.arange(len(y_test)), "true_label": y_test, "predicted_label": preds[name],
                  "prob_positive": probs[name]}).to_csv(OUT / "predictions" / f"{name}_test_predictions.csv", index=False)

metrics_df = pd.DataFrame(metrics)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
metrics_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, name in zip(axes, ckpts):
    ax.imshow(plt.imread(OUT / "plots" / f"confusion_{name}.png")); ax.axis("off")
fig.tight_layout(); plt.show()
ev.plot_curves(probs, y_test, OUT / "plots" / "roc_pr_calibration.png")
Image(filename=str(OUT / "plots" / "roc_pr_calibration.png"))

### 95% bootstrap confidence intervals (1,000 resamples) and paired McNemar tests

In [ ]:
ci = pd.DataFrame({n: {f"{k}": f"{m[k if k != 'macro_f1' else 'f1_macro']:.4f} [{m[k + '_ci95_lower']:.4f}, {m[k + '_ci95_upper']:.4f}]"
                       for k in ["accuracy", "macro_f1", "mcc"]} for n, m in metrics.items()}).T
display(ci)
mc = {f"baseline vs {n}": ev.mcnemar(y_test, preds["baseline"], preds[n]) for n in ["experimental_1", "experimental_2"]}
mc["experimental_1 vs experimental_2"] = ev.mcnemar(y_test, preds["experimental_1"], preds["experimental_2"])
mcnemar_df = pd.DataFrame(mc).T
mcnemar_df

### Robustness: macro-F1 and error rate per data slice

In [ ]:
slice_rows = []
for name, sl in per_slice.items():
    for s, v in sl.items():
        slice_rows.append({"model": name, "slice": s, **v})
slice_df = pd.DataFrame(slice_rows)
display(slice_df.pivot(index="slice", columns="model", values="macro_f1").join(
        slice_df.pivot(index="slice", columns="model", values="error_rate"), lsuffix="_macroF1", rsuffix="_errRate")
        .join(slice_df[slice_df.model == "baseline"].set_index("slice")["n"]))

### Sanity check: did the from-scratch embeddings learn sentiment?
Cosine nearest neighbours in the embedding table of the best model.

In [ ]:
best = max(summaries, key=lambda n: summaries[n]["best_val_macro_f1"])
print("best model by validation macro-F1:", best)
model, _ = load_model(ckpts[best], device)
E = torch.nn.functional.normalize(model.embedding.weight.detach().float().cpu(), dim=1)
itos = list(data["stoi"])
for w in ["great", "terrible", "delicious", "rude", "never", "recommend"]:
    if w in data["stoi"]:
        sims = E @ E[data["stoi"][w]]
        nn_ids = sims.topk(9).indices[1:].tolist()
        print(f"{w:>10}: {', '.join(itos[i] for i in nn_ids)}")

## Error review candidates (20 errors from the best model)
Chosen automatically on the test set: 5 confident false positives, 5 confident false negatives, 5 near-threshold errors (P≈0.5),
and 5 confident errors from the slice with the highest error rate. **The error types and proposed fixes are my own manual review, in `failure_analysis.md`.**
For experimental_2, the tokens with the highest attention weight are shown too.

In [ ]:
rows, worst_slice = ev.error_candidates(y_test, probs[best], preds[best], data["raw_text"]["test"], slices)
att_model, _ = load_model(ckpts["experimental_2"], device)
for r in rows:
    i = r["test_index"]
    ids = torch.from_numpy(data["ids"]["test"][i:i + 1]).long().to(device)
    ln = torch.from_numpy(data["lengths"]["test"][i:i + 1])
    with torch.no_grad():
        _, att = att_model(ids, ln, return_attention=True)
    a = att[0, : max(1, int(ln[0]))].cpu().numpy()
    toks = [itos[t] for t in data["ids"]["test"][i, : len(a)]]
    r["exp2_top_attention_tokens"] = ", ".join(f"{toks[j]}({a[j]:.2f})" for j in np.argsort(-a)[:5])
err_df = pd.DataFrame(rows)
err_df.to_csv(OUT / "error_review" / f"error_candidates_{best}.csv", index=False)
print("worst slice:", worst_slice)
with pd.option_context("display.max_colwidth", 400):
    display(err_df[["category", "true_label", "predicted_label", "prob_positive", "exp2_top_attention_tokens", "text"]])

## Save metrics report, run summary and environment

In [ ]:
rows = []
for name, m in metrics.items():
    for k, v in m.items():
        rows.append({"model": name, "metric": k, "value": v, "notes": "test set" + (" (smoke)" if SMOKE else "")})
    for s, v in per_slice[name].items():
        rows.append({"model": name, "metric": f"slice_macro_f1[{s}]", "value": v["macro_f1"], "notes": f"n={v['n']}"})
        rows.append({"model": name, "metric": f"slice_error_rate[{s}]", "value": v["error_rate"], "notes": f"n={v['n']}"})
    rows.append({"model": name, "metric": "hardware", "value": summaries[name]["hardware"].get("gpu"),
                 "notes": summaries[name]["hardware"]["cpu"]})
for pair, r in mc.items():
    rows.append({"model": pair, "metric": "mcnemar_p_value", "value": r["p_value"],
                 "notes": f"discordant: {r['a_right_b_wrong']} / {r['a_wrong_b_right']}"})
report_path = OUT / "metrics_report.csv" if SMOKE else TASK_DIR / "metrics_report.csv"
pd.DataFrame(rows).to_csv(report_path, index=False)

run_summary = {"smoke": SMOKE, "hardware": HW, "summaries": summaries, "test_metrics": metrics,
               "per_slice": per_slice, "mcnemar": mc, "best_model_by_val": best, "preprocessing": meta}
with open(OUT / "run_summary.json", "w") as f:
    json.dump(run_summary, f, indent=2, default=float)
env = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(OUT / "environment.txt").write_text(f"python {platform.python_version()}\ntorch {torch.__version__}\n{env}")
print("wrote", report_path.relative_to(TASK_DIR), "and", (OUT / "run_summary.json").relative_to(TASK_DIR))

# Reproducibility manifest: which checkpoint + raw log produced which reported number.
if not SMOKE:
    REPO = TASK_DIR.parent.parent
    man_dir = REPO / "reproducibility" / "manifests" / "dipin"
    manifest = {"task": "task2_sentiment", "member": "dipin", "config": "task2_sentiment/dipin/configs/task2_config.json",
                "notebook": "task2_sentiment/dipin/src/task2_dipin.ipynb", "metrics_file": "task2_sentiment/dipin/metrics_report.csv",
                "run_summary": "task2_sentiment/dipin/outputs/run_summary.json", "environment": "reproducibility/manifests/dipin/environment_task2.txt",
                "hardware": HW, "runs": {}}
    for n, s in summaries.items():
        manifest["runs"][n] = {"run_id": s["run_id"], "raw_log": s["raw_log"], "checkpoint": s["checkpoint"],
                               "best_epoch": s["best_epoch"], "test_accuracy": metrics[n]["accuracy"],
                               "test_macro_f1": metrics[n]["f1_macro"], "test_mcc": metrics[n]["mcc"]}
    (man_dir / "task2_manifest.json").write_text(json.dumps(manifest, indent=2))
    (man_dir / "environment_task2.txt").write_text((OUT / "environment.txt").read_text())
    print(json.dumps(manifest["runs"], indent=2))